# 3. Feature Engineering

Six augmentations applied to the raw reviews to enable downstream analysis:

| # | Augmentation | Script | Output Table |
|---|---|---|---|
| 3.1 | Per-Review Derived Features | `clean_data.py` | `review_features` |
| 3.2 | Sentiment Classification | `compute_sentiment.py` | `review_features.sentiment_*` |
| 3.3 | Topic Classification | `label_topics_with_llm.py` | `review_topics` |
| 3.4 | Health Score | `compute_health_score.py` | `app_weekly_metrics` |
| 3.5 | Stock Prices | `fetch_stock_prices.py` | `stock_prices`, `stock_tickers` |
| 3.6 | User-Level Features | `build_user_features.py` | `user_features` |

Each augmentation is run once to populate its target table. This notebook documents the rationale and shows sample output.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

import sqlite3
from pathlib import Path

# SQLite connection. The repo ships with the populated DB.
DB_PATH = Path("..") / "data" / "processed" / "reviews.db"
engine = sqlite3.connect(DB_PATH)

## 3.1 Per-Review Derived Features

**Script:** `clean_data.py`  
**Output:** `review_features` table

For each review, compute structural features not in the raw Google Play data: text length, word count, language, emoji count, day of week, hour of day, and whether the developer replied. Time-to-reply hours computed where applicable.

In [2]:
features_sample = pd.read_sql("""
SELECT review_id, text_length, word_count, language, 
    emoji_count, day_of_week, hour_of_day, has_developer_reply
FROM review_features
LIMIT 5
""", engine)
features_sample

,review_id,text_length,word_count,language,emoji_count,day_of_week,hour_of_day,has_developer_reply
0,97e9139b-cc46-4c80-8f2e-3c706afe17be,43,8,he,0,6,19,1
1,5c97e0fd-65d0-464e-addd-5640c3705537,241,41,he,0,1,19,1
2,6b6829b9-b468-4c3e-b7a0-bf910423f3e0,21,4,he,0,1,10,1
3,37abda91-bfcb-4db5-9280-3e3fd22ad5e7,173,31,he,0,6,15,1
4,ac60aa3d-2e26-4495-85a4-1cdad681062c,6,1,he,0,6,8,1


## 3.2 Sentiment Classification

**Script:** `compute_sentiment.py`  
**Model:** DictaBERT (Hebrew-specific BERT, ~500MB)  
**Output:** `review_features.sentiment_label`, `sentiment_score`

Captures polarity beyond the 1-5 star rating. A user can give 5 stars and write a complaint, or 1 star and mention a positive aspect. Sentiment adds a second signal independent of the star rating.

Restricted to Hebrew reviews with text length > 2.

In [3]:
sentiment_dist = pd.read_sql("""
SELECT sentiment_label, COUNT(*) AS n
FROM review_features
WHERE sentiment_label IS NOT NULL
GROUP BY sentiment_label
""", engine)
total = sentiment_dist['n'].sum()
sentiment_dist['pct'] = (sentiment_dist['n'] / total * 100).round(1)
print(f"Classified: {total:,} reviews")
sentiment_dist

Classified: 45,883 reviews


,sentiment_label,n,pct
0,Negative,14682,32.0
1,Neutral,6485,14.1
2,Positive,24716,53.9


## 3.3 Topic Classification

**Script:** `label_topics_with_llm.py`  
**Model:** Claude Haiku 4.5 (Anthropic API)  
**Output:** `review_topics` table

Classify each review into one of 10 business-relevant topics: stability, security, login_auth, performance, customer_service, ui_ux, features, fees, praise, other. LLM handles context better than keyword matching, e.g., distinguishing "the app is fast" (performance praise) from "I want fast transfers" (feature request).

Sampling stratified by sentiment to ensure balanced topic coverage.

In [4]:
topics_dist = pd.read_sql("""
SELECT topic, COUNT(*) AS n
FROM review_topics
GROUP BY topic
ORDER BY n DESC
""", engine)
total = topics_dist['n'].sum()
topics_dist['pct'] = (topics_dist['n'] / total * 100).round(1)
print(f"Labeled: {total:,} reviews across {len(topics_dist)} topics")
topics_dist

Labeled: 31,109 reviews across 10 topics


,topic,n,pct
0,praise,12340,39.7
1,stability,6558,21.1
2,features,3461,11.1
3,ui_ux,2357,7.6
4,customer_service,1853,6.0
5,login_auth,1802,5.8
6,performance,1114,3.6
7,other,989,3.2
8,security,373,1.2
9,fees,262,0.8


## 3.4 Health Score

**Script:** `compute_health_score.py`  
**Output:** `app_weekly_metrics` table

A weekly composite metric per app for trend tracking. Combines four signals:

- `avg_rating`: average star score
- `sentiment_balance`: positive ratio minus negative ratio
- `momentum`: change in `avg_rating` over a 4-week window
- `volume_normalized`: review count scaled within each app

The composite acts as a single dial for "how is this app doing this week".

In [5]:
health = pd.read_sql("""
SELECT app_id, year_week, review_count,
    ROUND(avg_rating, 2) AS avg_rating,
    ROUND(sentiment_balance, 2) AS sentiment_balance,
    ROUND(momentum, 2) AS momentum,
    ROUND(health_score, 1) AS health_score
FROM app_weekly_metrics
ORDER BY year_week DESC, app_id
LIMIT 8
""", engine)
health

,app_id,year_week,review_count,avg_rating,sentiment_balance,momentum,health_score
0,com.fibi.nativeapp,2026-19,4,4.00,0.25,-1.00,53.5
1,com.ideomobile.discount,2026-19,2,2.50,-1.00,1.25,28.2
2,com.ideomobile.hapoalim,2026-19,2,1.00,-1.00,-3.20,7.0
3,com.ideomobile.leumicard,2026-19,1,1.00,-1.00,-1.33,6.7
4,com.isracard.hatavot,2026-19,2,1.50,-1.00,-3.50,6.4
5,com.fibi.nativeapp,2026-18,6,4.83,1.00,0.08,79.1
6,com.ideomobile.discount,2026-18,5,3.00,-0.20,1.67,46.5
7,com.ideomobile.hapoalim,2026-18,3,4.67,0.67,2.38,78.5


## 3.5 Stock Prices

**Script:** `fetch_stock_prices.py`  
**Source:** Yahoo Finance (yfinance), TASE listings  
**Output:** `stock_tickers`, `stock_prices` tables

Daily stock prices for the six TASE-listed companies behind our apps: Hapoalim (POLI.TA), Leumi (LUMI.TA, parent of Pepper), Discount (DSCT.TA, parent of Mercantile and Cal), Mizrahi-Tefahot (MZTF.TA), F.I.B.I. Holdings (FIBIH.TA), and Isracard (ISCD.TA).

Enables a cross-source question: does user satisfaction track stock movement?

In [6]:
stocks = pd.read_sql("""
SELECT ticker, [date],
       ROUND([close], 2) AS close_price,
       ROUND(daily_return, 4) AS daily_return
FROM stock_prices
WHERE [date] >= '2025-01-01'
ORDER BY ticker, [date]
""", engine)
print(f"Total trading days in DB: {pd.read_sql('SELECT COUNT(*) AS n FROM stock_prices', engine).iloc[0,0]:,}")
stocks.head(10)

Total trading days in DB: 17,821


,ticker,date,close_price,daily_return
0,DSCT.TA,2025-01-01,2529.40,0.0152
1,DSCT.TA,2025-01-02,2482.41,-0.0186
2,DSCT.TA,2025-01-06,2567.39,0.0342
3,DSCT.TA,2025-01-07,2556.39,-0.0043
4,DSCT.TA,2025-01-08,2518.40,-0.0149
5,DSCT.TA,2025-01-09,2523.40,0.0020
6,DSCT.TA,2025-01-13,2578.39,0.0218
7,DSCT.TA,2025-01-14,2603.38,0.0097
8,DSCT.TA,2025-01-15,2616.38,0.0050
9,DSCT.TA,2025-01-16,2645.37,0.0111


## 3.6 User-Level Features

**Script:** `build_user_features.py`  
**Output:** `user_features` table

Aggregate per-user features for clustering and prediction: number of reviews, average score, score variance, average review length, lifespan, review rate, sentiment ratios.

Key design choice: filter shared usernames by frequency. Names that appear more than 50 times are treated as defaults (like `משתמש Google`, 20,613 reviews) and excluded. This is more general than exact-string matching since it catches any default Google Play might assign.

**Caveat:** Even after frequency filtering, common Hebrew names (Yossi Cohen, Haim Cohen, etc.) can conflate multiple distinct people sharing a real name. Google Play does not expose stable user IDs publicly, so this is a structural limit of the data. Aggregate patterns at the app or segment level are unaffected. User-level analysis should be read as approximate.

In [7]:
users = pd.read_sql("""
SELECT user_name, total_reviews, apps_reviewed,
    ROUND(avg_score, 2) AS avg_score,
    ROUND(std_score, 2) AS std_score,
    ROUND(avg_review_length, 1) AS avg_length,
    dominant_sentiment
FROM user_features
ORDER BY total_reviews DESC
LIMIT 5
""", engine)
print(f"Total real users: {pd.read_sql('SELECT COUNT(*) AS n FROM user_features', engine).iloc[0,0]:,}")
users

Total real users: 1,974


,user_name,total_reviews,apps_reviewed,avg_score,std_score,avg_length,dominant_sentiment
0,יוסי כהן,14,7,4.07,1.69,42.8,Positive
1,חיים כהן,12,6,4.08,1.56,47.6,Positive
2,דוד כהן,11,5,3.18,2.09,61.8,Negative
3,יעקב כהן,11,7,4.36,1.43,21.7,Positive
4,משה דובק,11,3,3.45,1.63,60.4,Neutral
